# OpenRouter Inference with Qwen 3.6 Plus

Practical inference examples using `qwen/qwen3.6-plus:free` — a 1M context, multimodal model available for free on OpenRouter.

**Requires:** `OPENROUTER_API_KEY` environment variable.

Covers: structured output, summarization, code generation, and reasoning with thinking tokens.

In [ ]:
import os
import json
import time
import requests

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY environment variable"

BASE_URL = "https://openrouter.ai/api/v1"
HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}
MODEL = "qwen/qwen3.6-plus:free"


def chat(messages, retries=5, delay=15, **kwargs):
    """Chat completion with rate limit retry."""
    payload = {"model": MODEL, "messages": messages, **kwargs}
    for attempt in range(retries):
        response = requests.post(
            f"{BASE_URL}/chat/completions",
            headers=HEADERS,
            json=payload,
        )
        if response.status_code == 429:
            wait = delay * (attempt + 1)
            print(f"Rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue
        response.raise_for_status()
        data = response.json()
        if "choices" not in data:
            print(f"Unexpected response (no choices), retrying... {data.get('error', '')}")
            time.sleep(delay)
            continue
        return data
    raise RuntimeError(f"Failed after {retries} attempts")


def ask(prompt, system=None, **kwargs):
    """Simple single-turn helper."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    data = chat(messages, **kwargs)
    return data["choices"][0]["message"]["content"]


print(f"Model: {MODEL}")
print(f"API key: ...{OPENROUTER_API_KEY[-4:]}")

## 1. Structured JSON Output

Ask the model to return structured data as JSON.

In [ ]:
result = ask(
    "List the 5 largest cities in Europe with their country and approximate population. "
    "Return ONLY valid JSON: an array of objects with keys: city, country, population.",
    system="You are a data assistant. Always respond with valid JSON, no markdown.",
)

# Parse and pretty-print
try:
    # Strip markdown code fences if present
    cleaned = result.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    data = json.loads(cleaned)
    print(json.dumps(data, indent=2))
except json.JSONDecodeError:
    print("Raw response (not valid JSON):")
    print(result)

## 2. Text Summarization

In [ ]:
article = """
The European Space Agency's Euclid space telescope has delivered its first scientific
results, mapping the distribution of dark matter across an unprecedented volume of the
universe. The data, collected over six months of observations, covers roughly 500 square
degrees of sky and reveals the cosmic web of dark matter filaments connecting galaxy
clusters. Scientists say the findings are consistent with the standard cosmological model
but hint at subtle tensions in the measured expansion rate of the universe. The Euclid
team plans to survey a total of 15,000 square degrees over the mission's six-year
lifetime, creating the largest 3D map of the cosmos ever assembled. The telescope uses
both visible and near-infrared imaging to measure the shapes and redshifts of billions
of galaxies, enabling precise measurements of how dark energy has influenced cosmic
structure formation over the past 10 billion years.
"""

summary = ask(
    f"Summarize this article in exactly 3 bullet points:\n\n{article}",
    system="You are a concise scientific editor.",
)
print(summary)

## 3. Code Generation

In [ ]:
code = ask(
    "Write a Python function that takes a list of integers and returns the longest "
    "increasing subsequence. Include type hints and a brief docstring. "
    "Return ONLY the function, no explanation.",
    system="You are a Python expert. Return only code, no markdown fences.",
)

# Strip markdown if present
cleaned = code.strip()
if cleaned.startswith("```"):
    cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]

print(cleaned)

# Execute it
exec(cleaned)
print("\n--- Test ---")
print(f"LIS of [10, 9, 2, 5, 3, 7, 101, 18]: {longest_increasing_subsequence([10, 9, 2, 5, 3, 7, 101, 18])}")

## 4. Reasoning with Thinking Tokens

Qwen 3.6 Plus supports `include_reasoning: true` which returns the model's chain-of-thought reasoning alongside the final answer.

In [ ]:
messages = [
    {"role": "user", "content": (
        "A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left? "
        "Think step by step."
    )}
]

data = chat(messages, include_reasoning=True)
choice = data["choices"][0]["message"]

# Show reasoning if available
if choice.get("reasoning"):
    print("=== Reasoning ===")
    print(choice["reasoning"])
    print()

print("=== Answer ===")
print(choice["content"])

print("\n--- Usage ---")
usage = data.get("usage", {})
print(f"Prompt: {usage.get('prompt_tokens', '?')} | Completion: {usage.get('completion_tokens', '?')} | Total: {usage.get('total_tokens', '?')}")

## 5. Translation Pipeline

In [ ]:
text = "The quick brown fox jumps over the lazy dog."
languages = ["German", "French", "Japanese", "Arabic"]

translations = ask(
    f"Translate the following text into {', '.join(languages)}.\n\n"
    f"Text: \"{text}\"\n\n"
    "Format as:\nLanguage: translation",
    system="You are a professional translator. Be accurate and natural.",
)

print(f"Original: {text}\n")
print(translations)